# Simulación Monte Carlo del juego de Yahtzee para dos jugadores

## 1. Introducción

El método de Monte Carlo es una técnica de simulación que utiliza números aleatorios para analizar sistemas donde existe incertidumbre. En esta actividad se aplica al juego clásico de Yahtzee, en el cual dos jugadores lanzan cinco dados convencionales de seis caras. Cada dado tiene una distribución de probabilidad uniforme, porque cada cara tiene la misma probabilidad de aparecer.

En cada turno, un jugador puede realizar hasta tres lanzamientos. Después de los lanzamientos, el resultado de los dados se evalúa según las categorías del juego, como unos, doses, tres iguales, cuatro iguales, full house, escalera pequeña, escalera grande, Yahtzee y chance.

El objetivo de la simulación es ejecutar muchas partidas completas para explotar los resultados aleatorios obtenidos, analizar puntajes, frecuencias de categorías, probabilidades de victoria y comportamiento general del sistema.

## 2. Distribución de probabilidad asociada

El experimento aleatorio base es el lanzamiento de un dado justo de seis caras. Por tanto, la variable aleatoria $X$ representa el valor obtenido en un dado:

$$X \in \{1,2,3,4,5,6\}$$

La distribución de probabilidad es uniforme discreta:

$$P(X=x)=\frac{1}{6}, \quad x \in \{1,2,3,4,5,6\}$$

Como en cada turno se lanzan cinco dados, el estado de los dados se representa como una lista de cinco valores. Cada lanzamiento es independiente de los anteriores, y todos los dados siguen la misma distribución uniforme.

## 3. Justificación del modelo de Markov

El juego puede interpretarse como un proceso de Markov porque el siguiente estado depende únicamente del estado actual y del resultado aleatorio del siguiente lanzamiento, no de toda la historia previa.

Para esta simulación, el estado del sistema puede definirse como:

$$S_t = (D_t, C_t, P_t, R_t)$$

Donde:

- $D_t$: valores actuales de los cinco dados.
- $C_t$: categorías ya utilizadas por el jugador.
- $P_t$: puntaje acumulado.
- $R_t$: número de lanzamiento dentro del turno.

La transición entre estados se produce cuando se relanzan algunos dados. Los nuevos valores se generan con distribución uniforme discreta. Por tanto, el estado futuro depende del estado presente y de la probabilidad del lanzamiento, cumpliendo la propiedad markoviana:

$$P(S_{t+1} \mid S_t, S_{t-1}, ..., S_0) = P(S_{t+1} \mid S_t)$$

En términos prácticos, si se conocen los dados actuales, las categorías disponibles y el número de lanzamientos restantes, no se necesita conocer toda la secuencia anterior de jugadas para decidir la siguiente transición.

## 4. Metodología de implementación

La metodología aplicada fue la siguiente:

1. Definir las reglas básicas del juego Yahtzee para dos jugadores.
2. Representar los dados como variables aleatorias uniformes entre 1 y 6.
3. Crear estructuras de datos para manejar jugadores, categorías, puntajes y resultados por turno.
4. Implementar funciones para calcular la puntuación de cada categoría.
5. Implementar una estrategia automática simple para decidir qué dados conservar entre lanzamientos.
6. Simular partidas completas de 13 turnos por jugador.
7. Ejecutar muchas partidas para aplicar Monte Carlo.
8. Generar tablas, estadísticas, gráficas y un archivo Excel con los resultados.

Nota: la estrategia del jugador es automática y heurística. No pretende ser una estrategia óptima profesional de Yahtzee, sino una estrategia suficiente para ejecutar la simulación y analizar resultados aleatorios.

## 5. Configuración inicial

In [1]:
%pip install -q pandas openpyxl matplotlib

import random
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set

import pandas as pd
import matplotlib.pyplot as plt

# Parámetros principales de la simulación
NUM_PARTIDAS = 1000
NUM_JUGADORES = 2
TURNOS_POR_JUGADOR = 13
DADOS_POR_TURNO = 5
CARAS_DADO = 6
MAX_LANZAMIENTOS = 3

# Semilla opcional para reproducibilidad.
# Puedes cambiarla o dejarla en None si deseas resultados diferentes en cada ejecución.
SEMILLA = 42
if SEMILLA is not None:
    random.seed(SEMILLA)

print("Configuración cargada correctamente.")


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Matplotlib is building the font cache; this may take a moment.


Configuración cargada correctamente.


## 6. Categorías y funciones de puntuación

In [ ]:
CATEGORIAS = [
    "unos", "doses", "treses", "cuatros", "cincos", "seises",
    "tres_iguales", "cuatro_iguales", "full_house",
    "escalera_pequena", "escalera_grande", "yahtzee", "chance"
]

NOMBRES_CATEGORIAS = {
    "unos": "Unos",
    "doses": "Doses",
    "treses": "Treses",
    "cuatros": "Cuatros",
    "cincos": "Cincos",
    "seises": "Seises",
    "tres_iguales": "Tres iguales",
    "cuatro_iguales": "Cuatro iguales",
    "full_house": "Full House",
    "escalera_pequena": "Escalera pequeña",
    "escalera_grande": "Escalera grande",
    "yahtzee": "Yahtzee",
    "chance": "Chance"
}

def lanzar_dados(cantidad: int = DADOS_POR_TURNO) -> List[int]:
    """Lanza 'cantidad' dados justos de seis caras con distribución uniforme discreta."""
    return [random.randint(1, CARAS_DADO) for _ in range(cantidad)]


def puntuacion_categoria(dados: List[int], categoria: str) -> int:
    """Calcula el puntaje de una lista de cinco dados para una categoría específica."""
    conteo = Counter(dados)
    total = sum(dados)
    valores_unicos = set(dados)

    if categoria == "unos":
        return dados.count(1) * 1
    if categoria == "doses":
        return dados.count(2) * 2
    if categoria == "treses":
        return dados.count(3) * 3
    if categoria == "cuatros":
        return dados.count(4) * 4
    if categoria == "cincos":
        return dados.count(5) * 5
    if categoria == "seises":
        return dados.count(6) * 6
    if categoria == "tres_iguales":
        return total if max(conteo.values()) >= 3 else 0
    if categoria == "cuatro_iguales":
        return total if max(conteo.values()) >= 4 else 0
    if categoria == "full_house":
        return 25 if sorted(conteo.values()) == [2, 3] else 0
    if categoria == "escalera_pequena":
        escaleras = [{1,2,3,4}, {2,3,4,5}, {3,4,5,6}]
        return 30 if any(e.issubset(valores_unicos) for e in escaleras) else 0
    if categoria == "escalera_grande":
        return 40 if valores_unicos in ({1,2,3,4,5}, {2,3,4,5,6}) else 0
    if categoria == "yahtzee":
        return 50 if max(conteo.values()) == 5 else 0
    if categoria == "chance":
        return total

    raise ValueError(f"Categoría no reconocida: {categoria}")


def evaluar_categorias_disponibles(dados: List[int], categorias_usadas: Set[str]) -> Dict[str, int]:
    """Evalúa todas las categorías todavía disponibles para un jugador."""
    return {
        categoria: puntuacion_categoria(dados, categoria)
        for categoria in CATEGORIAS
        if categoria not in categorias_usadas
    }

print("Categorías y funciones de puntuación listas.")

## 7. Estrategia automática de relanzamiento

In [2]:
def seleccionar_dados_a_conservar(dados: List[int], categorias_usadas: Set[str]) -> List[int]:
    """
    Estrategia automática simple para decidir qué dados conservar.

    Prioridad:
    1. Si ya hay Yahtzee, conservar todo.
    2. Si hay 4, 3 o 2 dados iguales, conservar el grupo más repetido.
    3. Si hay una posible escalera grande o pequeña, conservar la secuencia más larga.
    4. Si no hay patrón claro, conservar dados altos para Chance.
    """
    conteo = Counter(dados)
    mas_comun_valor, mas_comun_cantidad = conteo.most_common(1)[0]

    # Si hay muchos repetidos y la categoría aún puede servir, conservar esos dados.
    if mas_comun_cantidad >= 2:
        return [d for d in dados if d == mas_comun_valor]

    # Buscar secuencia útil para escaleras.
    valores = sorted(set(dados))
    posibles_secuencias = [
        [1, 2, 3, 4, 5],
        [2, 3, 4, 5, 6],
        [1, 2, 3, 4],
        [2, 3, 4, 5],
        [3, 4, 5, 6],
    ]
    mejor_secuencia = []
    for secuencia in posibles_secuencias:
        presente = [v for v in secuencia if v in valores]
        if len(presente) > len(mejor_secuencia):
            mejor_secuencia = presente

    if len(mejor_secuencia) >= 3:
        conservados = []
        usados = set()
        for d in dados:
            if d in mejor_secuencia and d not in usados:
                conservados.append(d)
                usados.add(d)
        return conservados

    # Conservar dados altos si no hay otra combinación clara.
    return [d for d in dados if d >= 5]


def elegir_mejor_categoria(dados: List[int], categorias_usadas: Set[str]) -> Tuple[str, int]:
    """Selecciona la categoría disponible con mayor puntaje inmediato."""
    puntajes = evaluar_categorias_disponibles(dados, categorias_usadas)
    mejor_categoria = max(puntajes, key=puntajes.get)
    return mejor_categoria, puntajes[mejor_categoria]

print("Estrategia automática definida.")

Estrategia automática definida.


## 8. Simulación de turnos, partidas y Monte Carlo

In [3]:
@dataclass
class Jugador:
    nombre: str
    puntajes: Dict[str, int] = field(default_factory=dict)
    categorias_usadas: Set[str] = field(default_factory=set)

    def __post_init__(self):
        self.puntajes = {categoria: 0 for categoria in CATEGORIAS}

    @property
    def total(self) -> int:
        return sum(self.puntajes.values())


def jugar_turno(jugador: Jugador, numero_partida: int, numero_turno: int) -> Dict:
    """
    Simula un turno completo para un jugador.
    Se respeta estrictamente el máximo de 3 lanzamientos por turno.
    """
    historial_lanzamientos = []

    # Lanzamiento 1 obligatorio
    dados = lanzar_dados(DADOS_POR_TURNO)
    historial_lanzamientos.append(dados.copy())

    # Lanzamientos 2 y 3 opcionales
    for lanzamiento_actual in range(2, MAX_LANZAMIENTOS + 1):
        dados_conservados = seleccionar_dados_a_conservar(dados, jugador.categorias_usadas)
        cantidad_a_relanzar = DADOS_POR_TURNO - len(dados_conservados)

        if cantidad_a_relanzar <= 0:
            break

        dados = sorted(dados_conservados + lanzar_dados(cantidad_a_relanzar))
        historial_lanzamientos.append(dados.copy())

    categoria, puntos = elegir_mejor_categoria(dados, jugador.categorias_usadas)
    jugador.puntajes[categoria] = puntos
    jugador.categorias_usadas.add(categoria)

    return {
        "Partida": numero_partida,
        "Turno": numero_turno,
        "Jugador": jugador.nombre,
        "Lanzamientos_Realizados": len(historial_lanzamientos),
        "Lanzamiento_1": str(historial_lanzamientos[0]) if len(historial_lanzamientos) >= 1 else "",
        "Lanzamiento_2": str(historial_lanzamientos[1]) if len(historial_lanzamientos) >= 2 else "",
        "Lanzamiento_3": str(historial_lanzamientos[2]) if len(historial_lanzamientos) >= 3 else "",
        "Dados_Finales": str(dados),
        "Categoria_Elegida": categoria,
        "Categoria_Elegida_Texto": NOMBRES_CATEGORIAS[categoria],
        "Puntuacion_Turno": puntos,
        "Puntuacion_Acumulada": jugador.total
    }


def jugar_partida(numero_partida: int) -> Tuple[Dict, List[Dict]]:
    """Simula una partida completa de 13 turnos por cada uno de los 2 jugadores."""
    jugador1 = Jugador("Jugador 1")
    jugador2 = Jugador("Jugador 2")
    detalle_turnos = []

    for turno in range(1, TURNOS_POR_JUGADOR + 1):
        detalle_turnos.append(jugar_turno(jugador1, numero_partida, turno))
        detalle_turnos.append(jugar_turno(jugador2, numero_partida, turno))

    if jugador1.total > jugador2.total:
        ganador = "Jugador 1"
    elif jugador2.total > jugador1.total:
        ganador = "Jugador 2"
    else:
        ganador = "Empate"

    resumen = {
        "Partida": numero_partida,
        "Puntaje_Jugador_1": jugador1.total,
        "Puntaje_Jugador_2": jugador2.total,
        "Diferencia_Puntos": abs(jugador1.total - jugador2.total),
        "Ganador": ganador,
        "Yahtzee_Jugador_1": jugador1.puntajes["yahtzee"],
        "Yahtzee_Jugador_2": jugador2.puntajes["yahtzee"],
        "Categorias_Usadas_J1": str(jugador1.categorias_usadas),
        "Categorias_Usadas_J2": str(jugador2.categorias_usadas)
    }

    return resumen, detalle_turnos


def simular_montecarlo(numero_partidas: int = NUM_PARTIDAS) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Ejecuta la simulación Monte Carlo de varias partidas completas."""
    resumen_partidas = []
    detalle_total = []

    for partida in range(1, numero_partidas + 1):
        resumen, detalle = jugar_partida(partida)
        resumen_partidas.append(resumen)
        detalle_total.extend(detalle)

    return pd.DataFrame(resumen_partidas), pd.DataFrame(detalle_total)

resumen_df, detalle_df = simular_montecarlo(NUM_PARTIDAS)

print(f"Simulación finalizada: {NUM_PARTIDAS} partidas completas.")
print(f"Registros de detalle generados: {len(detalle_df)}")
resumen_df.head()

NameError: name 'CATEGORIAS' is not defined

## 9. Validación del límite de lanzamientos

In [ ]:
# Validación técnica exigida por la instrucción: cada turno debe tener máximo 3 lanzamientos.
max_lanzamientos_observado = detalle_df["Lanzamientos_Realizados"].max()
min_lanzamientos_observado = detalle_df["Lanzamientos_Realizados"].min()

print("Mínimo de lanzamientos observados en un turno:", min_lanzamientos_observado)
print("Máximo de lanzamientos observados en un turno:", max_lanzamientos_observado)

assert max_lanzamientos_observado <= 3, "Error: hay turnos con más de 3 lanzamientos."
assert min_lanzamientos_observado >= 1, "Error: hay turnos sin lanzamiento."

print("Validación correcta: todos los turnos tienen entre 1 y 3 lanzamientos.")

## 10. Resultados estadísticos

In [ ]:
total_partidas = len(resumen_df)
conteo_ganadores = resumen_df["Ganador"].value_counts().reindex(["Jugador 1", "Jugador 2", "Empate"], fill_value=0)
porcentaje_ganadores = (conteo_ganadores / total_partidas * 100).round(2)

resumen_estadistico = pd.DataFrame({
    "Indicador": [
        "Total de partidas simuladas",
        "Victorias Jugador 1",
        "Victorias Jugador 2",
        "Empates",
        "Probabilidad victoria Jugador 1 (%)",
        "Probabilidad victoria Jugador 2 (%)",
        "Probabilidad empate (%)",
        "Promedio puntaje Jugador 1",
        "Promedio puntaje Jugador 2",
        "Puntaje máximo Jugador 1",
        "Puntaje máximo Jugador 2",
        "Puntaje mínimo Jugador 1",
        "Puntaje mínimo Jugador 2",
        "Diferencia promedio de puntos"
    ],
    "Valor": [
        total_partidas,
        int(conteo_ganadores["Jugador 1"]),
        int(conteo_ganadores["Jugador 2"]),
        int(conteo_ganadores["Empate"]),
        float(porcentaje_ganadores["Jugador 1"]),
        float(porcentaje_ganadores["Jugador 2"]),
        float(porcentaje_ganadores["Empate"]),
        round(resumen_df["Puntaje_Jugador_1"].mean(), 2),
        round(resumen_df["Puntaje_Jugador_2"].mean(), 2),
        int(resumen_df["Puntaje_Jugador_1"].max()),
        int(resumen_df["Puntaje_Jugador_2"].max()),
        int(resumen_df["Puntaje_Jugador_1"].min()),
        int(resumen_df["Puntaje_Jugador_2"].min()),
        round(resumen_df["Diferencia_Puntos"].mean(), 2)
    ]
})

resumen_estadistico

## 11. Frecuencia de categorías elegidas

In [ ]:
frecuencia_categorias = (
    detalle_df.groupby(["Categoria_Elegida_Texto"])
    .size()
    .reset_index(name="Frecuencia")
    .sort_values("Frecuencia", ascending=False)
)

frecuencia_categorias["Porcentaje"] = (frecuencia_categorias["Frecuencia"] / len(detalle_df) * 100).round(2)
frecuencia_categorias

## 12. Gráficas de resultados

In [ ]:
# Gráfica 1: victorias por jugador
plt.figure(figsize=(8, 5))
plt.bar(conteo_ganadores.index, conteo_ganadores.values)
plt.title("Victorias por jugador en la simulación Monte Carlo")
plt.xlabel("Resultado")
plt.ylabel("Cantidad de partidas")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# Gráfica 2: distribución de puntajes
plt.figure(figsize=(9, 5))
plt.hist(resumen_df["Puntaje_Jugador_1"], bins=30, alpha=0.6, label="Jugador 1")
plt.hist(resumen_df["Puntaje_Jugador_2"], bins=30, alpha=0.6, label="Jugador 2")
plt.title("Distribución de puntajes finales")
plt.xlabel("Puntaje final")
plt.ylabel("Frecuencia")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# Gráfica 3: promedio de puntajes
promedios = [resumen_df["Puntaje_Jugador_1"].mean(), resumen_df["Puntaje_Jugador_2"].mean()]

plt.figure(figsize=(7, 5))
plt.bar(["Jugador 1", "Jugador 2"], promedios)
plt.title("Comparación de puntajes promedio")
plt.xlabel("Jugador")
plt.ylabel("Puntaje promedio")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 13. Exportación a Excel

In [ ]:
# Cambia estos valores por los datos definitivos del estudiante o grupo.
APELLIDO = "Ardila"
NOMBRE = "JuanCarlos"

nombre_excel = f"{APELLIDO}_{NOMBRE}_Resultados_Montecarlo.xlsx"

modelo_probabilidad = pd.DataFrame({
    "Variable": ["Cara 1", "Cara 2", "Cara 3", "Cara 4", "Cara 5", "Cara 6"],
    "Probabilidad": [1/6] * 6,
    "Distribucion": ["Uniforme discreta"] * 6
})

with pd.ExcelWriter(nombre_excel, engine="openpyxl") as writer:
    resumen_df.to_excel(writer, sheet_name="Resumen_Partidas", index=False)
    detalle_df.to_excel(writer, sheet_name="Detalle_Partidas", index=False)
    resumen_estadistico.to_excel(writer, sheet_name="Resumen_Estadistico", index=False)
    frecuencia_categorias.to_excel(writer, sheet_name="Frecuencia_Categorias", index=False)
    modelo_probabilidad.to_excel(writer, sheet_name="Modelo_Probabilidad", index=False)

print(f"Archivo Excel generado: {nombre_excel}")

# Descarga opcional en Google Colab.
# Ejecuta estas dos líneas si quieres descargar el Excel automáticamente:
# from google.colab import files
# files.download(nombre_excel)

## 14. Resultados redactados para el documento

Durante la simulación Monte Carlo se ejecutaron múltiples partidas completas del juego Yahtzee para dos jugadores. Cada partida estuvo compuesta por 13 turnos por jugador y cada turno permitió hasta tres lanzamientos de cinco dados. Los dados fueron modelados mediante una distribución uniforme discreta, donde cada cara del dado tiene probabilidad de 1/6.

Los resultados obtenidos muestran variabilidad en los puntajes finales debido al componente aleatorio propio del juego. Esta variabilidad confirma la pertinencia del método de Monte Carlo, ya que permite repetir el experimento muchas veces para analizar tendencias generales y no depender de una única partida aislada.

A partir de la simulación se calcularon indicadores como número de victorias por jugador, empates, puntajes promedio, puntajes máximos, puntajes mínimos, diferencia promedio de puntos y frecuencia de categorías elegidas. Estos resultados permiten explotar los datos aleatorios generados por el programa y obtener una visión estadística del comportamiento del juego.

## 15. Conclusiones

El método de Monte Carlo permitió simular el juego Yahtzee mediante la generación repetida de resultados aleatorios, usando dados convencionales de seis caras con distribución uniforme. La simulación permitió observar cómo varían los puntajes y los ganadores entre partidas, aun cuando ambos jugadores tienen las mismas condiciones iniciales.

La estructura de datos utilizada permitió manejar adecuadamente la información del juego: jugadores, turnos, lanzamientos, categorías usadas, puntajes por turno, puntajes acumulados y resultados finales. Esto facilitó la explotación de los resultados mediante tablas, estadísticas y gráficas.

El modelo también puede justificarse desde la perspectiva de Markov, porque cada nuevo estado del juego depende del estado actual —dados, categorías disponibles, puntaje y lanzamientos restantes— y del resultado aleatorio del siguiente lanzamiento. No es necesario conocer toda la historia previa para determinar la transición siguiente.

En conclusión, la aplicación desarrollada cumple con el objetivo de comprender y aplicar el método de Monte Carlo en una simulación ejecutable, permitiendo analizar resultados aleatorios y generar evidencia cuantitativa para el informe académico.

## 16. Estructura sugerida para GitHub

Para cumplir el criterio de carga en GitHub, se recomienda organizar el repositorio así:

```text
montecarlo-yahtzee/
│
├── README.md
├── Apellido_Nombre_Montecarlo.ipynb
├── Apellido_Nombre_Resultados_Montecarlo.xlsx
├── codigo_fuente.py
└── documento/
    └── Apellido_Nombre_Montecarlo.docx
```

Contenido mínimo del README.md:

```text
# Simulación Monte Carlo - Yahtzee

Este proyecto implementa una simulación Monte Carlo del juego Yahtzee clásico para dos jugadores.

## Objetivo
Comprender y aplicar el método de Monte Carlo mediante una aplicación que simula lanzamientos aleatorios de dados y analiza los resultados obtenidos.

## Distribución de probabilidad
Cada dado sigue una distribución uniforme discreta entre 1 y 6. Cada cara tiene probabilidad 1/6.

## Modelo de Markov
El sistema puede interpretarse como un proceso de Markov porque el siguiente estado depende del estado actual y del resultado aleatorio del siguiente lanzamiento.

## Ejecución
Abrir el archivo .ipynb en Google Colab y ejecutar todas las celdas.

## Resultados
El programa genera estadísticas, gráficas y un archivo Excel con el detalle de las partidas simuladas.
```